<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day10-discussion-1.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 10, Segment 1 discussion — 2D convolution, and a real image-scale parameter count

The main notebook demonstrated translation equivariance with a **1D** convolution on a toy signal. Two extensions here: (1) the same equivariance property, verified numerically, for a **2D** convolution on a toy image — the case that actually motivated CNNs (ImageNet-style image classification); and (2) a real, computed parameter count for a realistic image size, to make "parameter explosion" from `day10.qmd` concrete rather than just asserted.

In [1]:
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)

## 1) Is a 2D convolution also translation-equivariant?

Same idea as the main notebook's 1D case, now on a small 2D "image": a single bright pixel in a 40x40 grid, and the same grid with that pixel shifted by (7, 7). An untrained 2D kernel should produce outputs that agree once you shift one of them by the same amount, away from the edges.

In [2]:
conv2d = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=5, padding=2, bias=False)

size = 40
image = torch.zeros(1, 1, size, size)
image[0, 0, 15, 15] = 1.0

shift = 7
shifted_image = torch.zeros(1, 1, size, size)
shifted_image[0, 0, 15 + shift, 15 + shift] = 1.0

with torch.no_grad():
    out = conv2d(image)[0, 0].numpy()
    out_shifted = conv2d(shifted_image)[0, 0].numpy()

out_rolled = np.roll(np.roll(out, shift, axis=0), shift, axis=1)
interior = (slice(10, 30), slice(10, 30))  # stay away from both edges in both dimensions
max_abs_diff = np.max(np.abs(out_rolled[interior] - out_shifted[interior]))
print(f"max |shift(conv2d(x)) - conv2d(shift(x))| over the interior region: {max_abs_diff:.2e}")
print("2D equivariant too (matches to floating-point precision away from the edges):", max_abs_diff < 1e-6)

max |shift(conv2d(x)) - conv2d(shift(x))| over the interior region: 0.00e+00
2D equivariant too (matches to floating-point precision away from the edges): True


## 2) A real parameter count: fully connected vs. convolutional, at image scale

`day10.qmd` claims a $224\times224\times3$ image would need "over 150,000" weights per hidden unit in a fully-connected layer. Compute the exact number, and compare it to a realistic small convolutional layer doing a comparable job (64 output channels, a $3\times3$ kernel).

In [3]:
H, W, C_in = 224, 224, 3
C_out = 64
kernel = 3

fc_weights_per_output_unit = H * W * C_in
fc_total_for_64_units = fc_weights_per_output_unit * C_out

conv_weights = kernel * kernel * C_in * C_out  # + C_out biases, ignored here for a clean comparison

print(f"Fully connected: {fc_weights_per_output_unit:,} weights per output unit")
print(f"Fully connected: {fc_total_for_64_units:,} weights total, for 64 output units")
print(f"Convolutional ({kernel}x{kernel} kernel, {C_out} output channels): {conv_weights:,} weights total")
print(f"Ratio: fully connected uses {fc_total_for_64_units / conv_weights:,.0f}x more weights")

Fully connected: 150,528 weights per output unit
Fully connected: 9,633,792 weights total, for 64 output units
Convolutional (3x3 kernel, 64 output channels): 1,728 weights total
Ratio: fully connected uses 5,575x more weights


## Discuss

1. The fully-connected layer's weight count doesn't even depend on the kernel size — every output unit looks at *every* input pixel. What would happen to the two counts above if the image were twice as large in each dimension (448x448)? Work out the new ratio before checking with code.
2. The convolutional layer's weight count above doesn't depend on the image size at all (224 or 448, same 1,728 weights). Why not — what is the convolution actually doing differently?

*One group presents.*

In [4]:
# Check your answer to question 1 here.
H2, W2 = 448, 448
fc_total_448 = H2 * W2 * C_in * C_out
print(f"Fully connected at 448x448: {fc_total_448:,} weights ({fc_total_448 / fc_total_for_64_units:.1f}x the 224x224 count)")
print(f"Convolutional at 448x448: {conv_weights:,} weights (unchanged)")

Fully connected at 448x448: 38,535,168 weights (4.0x the 224x224 count)
Convolutional at 448x448: 1,728 weights (unchanged)
